# Tópicos de Ingeniería Eléctrica II
## Semana 2 · Introducción a Python para adquisición y procesamiento de señales

**Universidad Autónoma de Ciudad Juárez — Instituto de Ingeniería y Tecnología**  
**Docente:** Dr. Ernesto Sifuentes de la Hoya  
**Modalidad:** cuaderno interactivo · 2 sesiones de 120 minutos

**Propósito.** Aprender los fundamentos de programación en Python que se utilizarán para adquirir, analizar, representar y documentar señales de sistemas electrónicos de medida.

> **Secuencia de aprendizaje:** comprender → programar → experimentar → analizar → comprobar.  
> Las señales de este cuaderno son **simuladas** para que pueda ejecutarse sin equipo electrónico. Se indicará explícitamente cuándo se utilice una medición real.

---

**Para comenzar:** en Google Colab, ejecute una celda con **▶** o con **Shift + Enter**. Modifique los valores y vuelva a ejecutar. Conserve una copia personal mediante **Archivo → Guardar una copia en Drive**.

### Objetivos de aprendizaje
Al finalizar la semana, el estudiante podrá:

1. Crear y ejecutar programas sencillos con variables, operadores, condiciones, ciclos y funciones.
2. Calcular una conversión elemental entre voltaje y cuentas de un convertidor analógico–digital (ADC).
3. Generar y procesar señales muestreadas con **NumPy**.
4. Representar señales e interpretar resultados mediante **Matplotlib**.
5. Organizar mediciones en una tabla y exportarlas a **CSV** mediante **Pandas**.
6. Explicar las decisiones del programa, verificar resultados y utilizar IA generativa como apoyo **sin sustituir el razonamiento propio**.

### Organización de sesiones

| Sesión | Tiempo | Trabajo | Producto |
|---|---:|---|---|
| **Clase 1** | 120 min | Entorno Colab, variables, operaciones, listas, decisiones, ciclos y funciones | Ejercicios A1–A4 |
| **Clase 2** | 120 min | Arreglos, señales muestreadas, gráficas, estadísticas, CSV y filtrado elemental | Mini práctica integradora y autoevaluación |

**Evidencia prevista:** ejercicios del cuaderno y archivo CSV de la mini práctica. **Entrega del Cuestionario L1:** clase 2 de esta semana. **Trabajo extra clase:** ejercicios de Python, para entregar en la clase 2 de la Semana 3.

## Clase 1 · Fundamentos de programación (120 min)
### 1. Conocer el entorno y ejecutar su primer programa (0–15 min)

Un cuaderno tiene **celdas de texto** (explicaciones, ecuaciones y actividades) y **celdas de código** (programas ejecutables). Python distingue mayúsculas y minúsculas. El símbolo `#` inicia un comentario que no se ejecuta.

**Experimente:** modifique el nombre de la variable y vuelva a ejecutar.

In [ ]:
# Primera ejecución
nombre_curso = "Tópicos de Ingeniería Eléctrica II"
print("Bienvenido a", nombre_curso)
print("Semana:", 2)
print("Python puede calcular, procesar datos y generar gráficas.")

### 2. Variables, tipos de datos y unidades (15–35 min)
Una variable conserva una referencia a un valor. En ingeniería es importante asignar **nombres descriptivos** y mantener claras las unidades.

- `int`: cuenta entera, por ejemplo una lectura de ADC.
- `float`: valor real, por ejemplo un voltaje.
- `str`: texto, por ejemplo el nombre del sensor.
- `bool`: condición lógica, por ejemplo una alarma.

**Ejemplo:** una lectura de tensión adquirida por un sistema de medida.

In [ ]:
# Ejemplo de datos de instrumentación
cuentas_adc = 2048           # número entero
voltaje_v = 1.65            # voltios
nombre_sensor = "Sensor 1"  # texto
alarma_activa = False       # valor lógico

for etiqueta, valor in [("Cuenta", cuentas_adc), ("Voltaje (V)", voltaje_v),
                        ("Sensor", nombre_sensor), ("Alarma", alarma_activa)]:
    print(f"{etiqueta}: {valor} (tipo {type(valor).__name__})")

### 3. Operadores y conversión de unidades (35–55 min)
El operador de potencia es `**`; la división decimal es `/`. Use paréntesis para dejar explícito el orden de las operaciones.

Ejemplo: si un sistema registra 125 muestras por segundo durante 8 s, entonces

$$N_{\mathrm{muestras}}=f_s\,t_{\mathrm{adquisición}}.$$

In [ ]:
frecuencia_muestreo_hz = 125
segundos = 8
numero_muestras = frecuencia_muestreo_hz * segundos
print("Número de muestras:", numero_muestras)
print("Tiempo entre muestras (s):", 1 / frecuencia_muestreo_hz)

**Ejercicio A1 · Cálculo de muestreo** (5 min)

Cambie el valor de `frecuencia_hz` a 200 y el tiempo a 12 s. Antes de ejecutar, prediga el número total de muestras y el intervalo de muestreo. Después compare sus cálculos con la ejecución.

In [ ]:
# A1 — Complete los valores y ejecute
frecuencia_hz = 125   # TODO: cambiar a 200
adquisicion_s = 8     # TODO: cambiar a 12
muestras = frecuencia_hz * adquisicion_s
intervalo_s = 1 / frecuencia_hz
print(f"Muestras: {muestras} | Intervalo: {intervalo_s:.4f} s")

### 4. ADC: de voltaje a cuenta digital (55–75 min)
Para un ADC ideal unipolar de $n$ bits y un intervalo $0\le V\le V_\mathrm{ref}$, una convención sencilla de conversión por redondeo es:

$$
\mathrm{cuenta}=\operatorname{redondear}\!\left(\frac{V}{V_\mathrm{ref}}(2^n-1)\right),
\qquad
\widehat{V}=\frac{\mathrm{cuenta}}{2^n-1}V_\mathrm{ref}.
$$

La separación entre niveles reconstruidos bajo esta convención es:

$$\Delta V =\frac{V_\mathrm{ref}}{2^n-1}.$$

**Atención:** es un **modelo didáctico**. La transferencia y el error del ADC de un ESP32 real dependen de su configuración, calibración y especificaciones. No se supone aquí linealidad ideal del hardware.

In [ ]:
# Conversión ideal ADC (NO corresponde a una lectura real)
v_ref = 3.3        # V
n_bits = 12
voltaje_entrada = 1.25  # V
codigo_max = 2**n_bits - 1
cuenta = round(max(0.0, min(v_ref, voltaje_entrada)) / v_ref * codigo_max)
voltaje_reconstruido = cuenta * v_ref / codigo_max
paso_v = v_ref / codigo_max
print(f"Cuenta ideal: {cuenta}")
print(f"Voltaje reconstruido: {voltaje_reconstruido:.5f} V")
print(f"Separación entre niveles: {paso_v*1000:.4f} mV")

**Ejercicio A2 · Resolución y error de cuantización**

Compare un ADC ideal de 10 bits con uno de 12 bits, utilizando $V_\mathrm{ref}=3.3$ V. Describa cuál representa incrementos de voltaje más pequeños y por qué.

In [ ]:
# A2 — Modifique los bits y los voltajes; explique su resultado
for bits in [10, 12]:
    niveles = 2**bits
    delta = 3.3 / (niveles - 1)
    print(f"{bits} bits: {niveles} códigos; paso reconstruido = {delta*1000:.4f} mV")
# TODO: agregue su conclusión como comentario en esta celda.

### 5. Listas y acceso a las mediciones (75–90 min)
La lista `[]` permite guardar una secuencia. La primera posición es el índice `0`. Las funciones `len`, `min`, `max` y `sum` son útiles para inspecciones iniciales.

In [ ]:
temperaturas_c = [24.8, 25.1, 25.4, 25.2, 25.7]
print("Primer dato:", temperaturas_c[0], "°C")
print("Número de muestras:", len(temperaturas_c))
print("Promedio:", round(sum(temperaturas_c) / len(temperaturas_c), 2), "°C")
# Pruebe cambiar el valor de una muestra y observe el promedio.

### 6. Condiciones, ciclos y funciones (90–115 min)

- `if`: ejecuta instrucciones cuando una condición se cumple.
- `for`: repite operaciones sobre valores de una secuencia.
- `def`: crea funciones reutilizables.

Una función es preferible a repetir el mismo cálculo en varios lugares. Para evaluar una alarma, distinga un umbral didáctico de un límite de seguridad real.

In [ ]:
# Ejemplo: alarma didáctica a 30 °C
umbral_c = 30.0
for temperatura in [27.1, 29.6, 30.2, 28.8]:
    if temperatura >= umbral_c:
        print(f"{temperatura:.1f} °C -> ALARMA")
    else:
        print(f"{temperatura:.1f} °C -> Normal")

def promedio_muestras(muestras):
    """Promedia una lista no vacía de mediciones."""
    if len(muestras) == 0:
        raise ValueError("Se necesita al menos una medición")
    return sum(muestras) / len(muestras)

print("Promedio de las temperaturas:", round(promedio_muestras(temperaturas_c), 2), "°C")

**Ejercicio A3 · Funciones**

Complete una función que devuelva la temperatura en kelvin a partir de grados Celsius usando $T_K=T_{^\circ C}+273.15$ y úsela para convertir 25 °C. Agregue una verificación del resultado esperado.

In [ ]:
# A3 — Complete la función

def a_kelvin(temperatura_c):
    # TODO: sustituya None por la expresión correspondiente
    return None

resultado_a3 = a_kelvin(25.0)
if resultado_a3 is None:
    print("A3 pendiente: complete la función y vuelva a ejecutar.")
else:
    print(f"25 °C equivalen a {resultado_a3:.2f} K")
    assert abs(resultado_a3 - 298.15) < 1e-9, "Revise la conversión de unidades"

**Ejercicio A4 · Interpretación del código**

¿Qué imprimirá el programa si se cambia `limite` a 28? ¿Qué ocurrirá cuando el valor sea exactamente igual al límite? Predígalo antes de ejecutar.

In [ ]:
limite = 30
lecturas = [27, 28, 30, 31]
conteo_alarmas = 0
for valor in lecturas:
    if valor >= limite:
        conteo_alarmas += 1
print("Número de lecturas por encima o iguales al límite:", conteo_alarmas)
# TODO: explique la diferencia entre > y >=.

### Cierre de la Clase 1 (115–120 min)

Explique en dos o tres oraciones cómo un programa puede convertir una tensión en un número, decidir si excede un umbral y presentar una alarma. Registre una duda para discutir en la siguiente clase.

---

## Clase 2 · Señales, gráficas y archivos (120 min)
### 7. NumPy: arreglos para señales muestreadas (0–25 min)

NumPy proporciona arreglos numéricos y operaciones vectorizadas. Para un muestreo uniforme:

$$t_k=\frac{k}{f_s},\quad k=0,1,\ldots,N-1.$$

Una señal armónica de prueba puede representarse como:

$$v(t)=V_0+A\sin(2\pi f_0 t)+\eta(t),$$

donde $\eta(t)$ representa ruido simulado. **No se trata de datos adquiridos mediante sensores reales.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Señal de prueba con semilla fija para reproducción exacta
fs = 20.0         # muestras/segundo
n_muestras = 200  # 10 segundos sin incluir el extremo t=10
f0 = 0.8          # Hz
v_base = 1.65     # V
amplitud = 0.70   # V
sigma_ruido = 0.08  # desviación estándar del ruido, V
rng = np.random.default_rng(2026)

t = np.arange(n_muestras) / fs
v_ideal = v_base + amplitud * np.sin(2 * np.pi * f0 * t)
v_medido = v_ideal + rng.normal(0.0, sigma_ruido, n_muestras)
print(f"Cantidad: {len(t)} muestras | Duración entre primera y última: {t[-1]:.2f} s")
print(f"Separación temporal: {t[1]-t[0]:.3f} s")

### 8. Visualización y lectura de gráficas (25–45 min)
Una gráfica de ingeniería debe incluir **ejes con unidades, leyenda y título descriptivo**. Comparamos una señal ideal con una medición simulada que contiene ruido.

In [ ]:
plt.figure(figsize=(9, 4.2))
plt.plot(t, v_ideal, label="Señal ideal", linewidth=2)
plt.plot(t, v_medido, label="Señal con ruido simulado", linewidth=1, alpha=0.8)
plt.title("Señal de tensión simulada")
plt.xlabel("Tiempo (s)")
plt.ylabel("Tensión (V)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

**Ejercicio B1 · Experimente con el muestreo**

Cambie la frecuencia de la señal `f0` de 0.8 a 2.0 Hz y vuelva a ejecutar las dos celdas anteriores. Mantenga la frecuencia de muestreo constante. ¿Cuántas oscilaciones observa aproximadamente en diez segundos? ¿Cambió el número de muestras?

### 9. Estadísticos y calidad de la señal (45–60 min)
Para $N$ valores $x_k$, se definen:

$$\overline{x}=\frac{1}{N}\sum_{k=0}^{N-1}x_k,$$

$$\sigma_x=\sqrt{\frac{1}{N}\sum_{k=0}^{N-1}(x_k-\overline{x})^2},$$

$$x_\mathrm{RMS}=\sqrt{\frac{1}{N}\sum_{k=0}^{N-1}x_k^2}.$$

El valor RMS **no equivale** a la desviación estándar si la señal tiene componente continua. En las celdas siguientes se utiliza la desviación estándar poblacional de la colección disponible (`ddof=0`).

In [ ]:
media_v = np.mean(v_medido)
desviacion_v = np.std(v_medido, ddof=0)
rms_v = np.sqrt(np.mean(v_medido**2))
rmse_ideal_v = np.sqrt(np.mean((v_medido - v_ideal)**2))
print(f"Media: {media_v:.4f} V")
print(f"Desviación estándar: {desviacion_v:.4f} V")
print(f"RMS total: {rms_v:.4f} V")
print(f"RMSE respecto a señal ideal conocida: {rmse_ideal_v:.4f} V")
# El RMSE anterior solo es posible porque en la simulación conocemos la señal ideal.

### 10. Tablas y archivos CSV (60–80 min)
Un archivo **CSV** permite trasladar lecturas a hojas de cálculo o a programas de análisis. Cada fila corresponde a una muestra con tiempo y valor.

En Colab, los archivos creados en la sesión son temporales; **descargue sus resultados antes de cerrar el entorno**.

In [ ]:
# Construir una tabla de mediciones simuladas
mediciones_df = pd.DataFrame({
    "tiempo_s": t,
    "tension_ideal_v": v_ideal,
    "tension_simulada_v": v_medido
})
print(mediciones_df.head(6).to_string(index=False))

nombre_csv = "semana02_senal_simulada.csv"
mediciones_df.to_csv(nombre_csv, index=False, float_format="%.6f")
print(f"Archivo CSV generado: {nombre_csv}")

In [ ]:
# Comprobar que podemos volver a leer el CSV
archivo_leido = pd.read_csv("semana02_senal_simulada.csv")
assert len(archivo_leido) == len(t), "No coincide el número de filas"
assert list(archivo_leido.columns) == ["tiempo_s", "tension_ideal_v", "tension_simulada_v"]
print("CSV reabierto correctamente. Dimensiones:", archivo_leido.shape)
print(archivo_leido.tail(3).to_string(index=False))

**Descarga en Colab.** Ejecute la celda siguiente si desea descargar el CSV a su computadora. En un Jupyter local la celda solamente mostrará su ubicación.

In [ ]:
from pathlib import Path
ruta_csv = Path("semana02_senal_simulada.csv").resolve()
try:
    from google.colab import files
except ImportError:
    print("Archivo disponible en:", ruta_csv)
else:
    # En Colab abre el diálogo de descarga cuando esta celda se ejecuta.
    files.download(str(ruta_csv))

### 11. Filtrado elemental: media móvil (80–95 min)
Un promedio móvil de $M$ muestras es

$$y[k]=\frac{1}{M}\sum_{j=0}^{M-1}x[k-j].$$

El filtrado reduce ciertas fluctuaciones, pero puede introducir **suavizado y retardo**. Aquí se usa una ventana **causal**: para los primeros puntos se promedian solo las muestras disponibles (`min_periods=1`).

In [ ]:
ventana = 9  # modifique a 3 o 15 y observe el efecto
v_filtrada = pd.Series(v_medido).rolling(window=ventana, min_periods=1).mean().to_numpy()
plt.figure(figsize=(9, 4.2))
plt.plot(t, v_medido, label="Señal con ruido simulado", alpha=0.5)
plt.plot(t, v_filtrada, label=f"Media móvil causal ({ventana} muestras)", linewidth=2)
plt.plot(t, v_ideal, label="Referencia ideal simulada", linestyle="--", linewidth=1.1)
plt.title("Efecto de la media móvil")
plt.xlabel("Tiempo (s)")
plt.ylabel("Tensión (V)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("RMSE (sin filtrar):", round(np.sqrt(np.mean((v_medido-v_ideal)**2)), 5), "V")
print("RMSE (filtrada):", round(np.sqrt(np.mean((v_filtrada-v_ideal)**2)), 5), "V")
# La comparación depende de la ventana, del ruido y de la frecuencia de la señal.

**Ejercicio B2 · Balance entre ruido y retardo**

Repita el resultado para ventanas de 3, 9 y 15 muestras. ¿Cuál reduce más las fluctuaciones? ¿Cuál deforma más los cambios rápidos? No suponga que la ventana de menor RMSE será siempre la óptima.

### 12. Mini práctica integradora: sensor de temperatura simulado (95–115 min)

**Problema:** un instrumento registra temperatura durante 30 s a 5 muestras por segundo. Se desea generar lecturas simuladas, mostrarlas y conservarlas en CSV.

Modelo sintético:

$$T(t)=T_a+\Delta T\left(1-e^{-t/\tau}\right)+\eta_T(t),$$

donde $T_a$ es la temperatura inicial, $\Delta T$ el incremento asintótico, $\tau$ una constante de tiempo y $\eta_T$ ruido simulado. **Este modelo es solamente un ejemplo de datos; no identifica todavía al ControlLab-ESP32 real.**

**Su trabajo:** ejecute, cambie `tau_s`, `sigma_t` y `umbral_c`, explique los cambios y exporte sus resultados.

In [ ]:
# Mini práctica — generación, análisis, gráfica y archivo CSV
rng_t = np.random.default_rng(7)
fs_temp_hz = 5.0
segundos_temp = 30.0
tiempo_temp = np.arange(0.0, segundos_temp, 1.0/fs_temp_hz)
t_amb_c = 24.0
delta_c = 14.0
tau_s = 8.0            # TODO: probar 4 y 12 s
sigma_t = 0.25         # TODO: probar 0.10 y 0.50 °C
umbral_c = 35.0        # umbral ilustrativo; NO es un límite de seguridad

temperatura_ideal = t_amb_c + delta_c*(1-np.exp(-tiempo_temp/tau_s))
temperatura_observada = temperatura_ideal + rng_t.normal(0, sigma_t, len(tiempo_temp))
alarma = temperatura_observada >= umbral_c

df_temp = pd.DataFrame({
    "tiempo_s": tiempo_temp,
    "temperatura_ideal_c": temperatura_ideal,
    "temperatura_simulada_c": temperatura_observada,
    "alarma_didactica": alarma
})

print("Muestras:", len(df_temp))
print("Temperatura media:", round(df_temp["temperatura_simulada_c"].mean(), 2), "°C")
print("Lecturas que activan umbral:", int(alarma.sum()))

plt.figure(figsize=(9, 4.2))
plt.plot(tiempo_temp, temperatura_ideal, label="Temperatura ideal simulada", linewidth=2)
plt.plot(tiempo_temp, temperatura_observada, label="Lectura con ruido simulado", alpha=0.7)
plt.axhline(umbral_c, linestyle="--", label="Umbral didáctico")
plt.xlabel("Tiempo (s)")
plt.ylabel("Temperatura (°C)")
plt.title("Adquisición de temperatura simulada")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

df_temp.to_csv("semana02_temperatura_simulada.csv", index=False, float_format="%.5f")
print("CSV creado: semana02_temperatura_simulada.csv")

**Comprobación guiada de la mini práctica**

1. ¿Qué parámetro controla la rapidez del calentamiento en el modelo?
2. ¿Qué efecto tiene aumentar `sigma_t` sobre la dispersión de los datos?
3. ¿Qué columnas deben acompañar siempre a una medición para que pueda interpretarse?
4. ¿Por qué no debemos presentar la curva anterior como un ensayo experimental de la planta física?

**Extensión voluntaria:** agregue una columna con temperatura filtrada y una segunda gráfica. Indique la ventana elegida y justifique su efecto.

### 13. Autoevaluación formativa (115–120 min)

Responda con una letra (`A`, `B` o `C`) en el diccionario siguiente. Ejecute y revise la retroalimentación. Las respuestas de este ejercicio **no reemplazan la evaluación oficial** de la asignatura.

| Ítem | Pregunta | Opciones |
|---|---|---|
| P1 | ¿Qué representa `fs`? | A: tiempo total · B: frecuencia de muestreo · C: amplitud |
| P2 | ¿Qué función calcula la media de un arreglo? | A: `np.mean` · B: `np.max` · C: `np.arange` |
| P3 | ¿Cuál formato conserva tablas de medición? | A: PNG · B: MP3 · C: CSV |
| P4 | ¿Qué se ejecuta con `for`? | A: ciclo · B: función definida · C: gráfica |
| P5 | ¿Qué describe el ruido en nuestra simulación? | A: medición real · B: fluctuación sintética · C: precisión certificada |
| P6 | ¿Qué unidad corresponde al eje horizontal de nuestras gráficas? | A: voltio · B: segundo · C: ampere |

In [ ]:
# Escriba una letra A, B o C para cada pregunta y ejecute.
respuestas = {"P1":"", "P2":"", "P3":"", "P4":"", "P5":"", "P6":""}

# Autoevaluación formativa (el estudiante puede consultar el código y aprender del resultado).
clave = {"P1":"B", "P2":"A", "P3":"C", "P4":"A", "P5":"B", "P6":"B"}
pistas = {
 "P1":"Se refiere al número de muestras obtenidas por segundo.",
 "P2":"NumPy implementa la operación promedio con una función específica.",
 "P3":"Es un formato de texto delimitado, apropiado para columnas de datos.",
 "P4":"Permite recorrer una secuencia de muestras.",
 "P5":"La señal fue generada por un modelo y una distribución aleatoria.",
 "P6":"El eje horizontal muestra la evolución temporal."
}
contestadas = sum(str(v).upper() in ('A','B','C') for v in respuestas.values())
aciertos = sum(str(respuestas[k]).upper() == clave[k] for k in clave)
print(f"Contestadas: {contestadas}/{len(clave)} | Aciertos: {aciertos}/{len(clave)}")
for k in clave:
    r = str(respuestas[k]).upper()
    if r not in ('A','B','C'):
        print(f"{k}: sin respuesta")
    elif r == clave[k]:
        print(f"{k}: correcto")
    else:
        print(f"{k}: revise. Pista: {pistas[k]}")

---
## Actividades con IA generativa · Tutor de aprendizaje

La IA puede ayudar a **entender, depurar y explicar**, pero el programa final y la interpretación de resultados deben ser producto del trabajo del estudiante.

**Actividad IA-1 · Preguntas guiadas**

> Actúe como tutor de programación para un estudiante de Ingeniería Eléctrica que empieza con Python. Necesito comprender cómo convertir cuentas de un ADC ideal de 12 bits a voltios. No me dé la solución completa al principio: formule una pregunta cada vez y revise mis unidades y razonamiento.

**Actividad IA-2 · Depuración orientada**

> Estoy desarrollando una práctica con NumPy y Matplotlib. Le compartiré mi código y el error obtenido. Explique qué significa el error, haga preguntas para localizar la causa y proponga una prueba mínima antes de mostrar una corrección.

**Actividad IA-3 · Interpretación técnica**

> Compare conceptualmente la media, desviación estándar y RMS para una señal con componente continua. Solicite que yo explique las diferencias usando los resultados de mi cuaderno; corrija mis argumentos con ejemplos físicos sencillos.

**Evidencia:** escriba brevemente qué pregunta formuló, qué sugerencia resultó útil, qué verificó por su cuenta y qué cambios realizó. Declare el uso de IA conforme a las indicaciones del curso. No comparta contraseñas, claves API ni datos personales en el tutor.

### Registro breve de aprendizaje (completar por el estudiante)

**Una dificultad que resolví:**

> ...

**Un resultado cuantitativo de mi práctica (con unidades):**

> ...

**Cómo verifiqué la respuesta de la IA:**

> ...

**Conclusión sobre muestreo, ruido o filtrado:**

> ...

---

## Trabajo extra clase

Realice los siguientes cambios y conserve sus resultados en el cuaderno:

1. **ADC:** compare los pasos de cuantización para 8, 10 y 12 bits con 3.3 V, expresándolos en mV.
2. **Señal:** genere 200 muestras con una nueva frecuencia fundamental y compare la señal ideal y la ruidosa.
3. **Estadística:** calcule media, desviación estándar y RMS para la nueva señal; explique qué mide cada parámetro.
4. **Filtrado:** compare tres ventanas de media móvil y argumente su elección.
5. **Datos:** exporte un CSV con tiempo, señal simulada y señal filtrada.
6. **IA:** documente una interacción de apoyo, incluyendo la comprobación técnica realizada por usted.

**Entrega:** clase 2 de la Semana 3. **Producto:** cuaderno `.ipynb` y CSV generado. La calificación y los criterios aplicables serán los comunicados por el docente.

## Notas sobre instrumentación real

El entorno Colab se ejecuta en una máquina remota. La lectura directa del puerto USB/COM de un ESP32 conectado a la computadora **no forma parte de este cuaderno**; cuando se incorpore hardware real, se utilizará una solución local o un mecanismo de comunicación compatible previamente verificado. No confunda datos simulados con adquisición experimental.

## Lecturas y recursos

- [Google Colab: guía oficial](https://research.google.com/colaboratory/faq.html?hl=es).
- [APMonitor, Programming for Engineers](https://apmonitor.com/che263/) — material complementario de Python y solución de problemas de ingeniería.
- [APMonitor, Process Dynamics and Control](https://apmonitor.com/pdc/) — referencia de organización: fundamentos, evaluación y experimentación.
- [Documentación de NumPy](https://numpy.org/doc/stable/).
- [Documentación de Pandas](https://pandas.pydata.org/docs/).
- [Documentación de Matplotlib](https://matplotlib.org/stable/).